# Satellite / fire exposure prep — v1

**Purpose:** Scaffold **data import**, **survey timing alignment**, and **preliminary linkage** keys before pulling MODIS/VIIRS (or other) gridded products. Later phases: inventory **derived fire products**, verify **coverage by place and timeframe**, and log product metadata in tables below.

**Survey calendar source:** `data/raw/VACS_survey_time.csv` (PI-maintained identifiable field periods; some rows are month-only or need guide confirmation).

**Sections:** §1 Load survey timing · §2 Parse / flag dates · §3 Linkage keys (country → ISO, future geo join) · §4 Product inventory (fill as research proceeds)

In [1]:
from __future__ import annotations

import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def _find_project_root() -> Path:
    """Resolve repo root using ``data/raw/VACS_survey_time.csv`` so the kernel cwd can vary."""
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents][:12]:
        candidate = d / "data" / "raw" / "VACS_survey_time.csv"
        if candidate.is_file():
            return d
    return cwd


ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
SURVEY_TIME_PATH = DATA_RAW / "VACS_survey_time.csv"

if not SURVEY_TIME_PATH.is_file():
    raise FileNotFoundError(
        "Could not find VACS_survey_time.csv under any parent of the kernel cwd.\n"
        f"  tried from: {Path.cwd().resolve()}\n"
        f"  expected at: {SURVEY_TIME_PATH}\n"
        "Fix: set the notebook kernel working directory to this repo, or open the project folder as the workspace root."
    )

print("ROOT =", ROOT)

ROOT = /Users/starsrain/research_side_projects_ipv


## §1 — Import survey timing table

The CSV may include a trailing column for notes. Normalize column names and inspect raw strings before parsing dates.

In [2]:
raw = pd.read_csv(SURVEY_TIME_PATH, dtype=str, keep_default_na=False)
raw.columns = [c.strip() if isinstance(c, str) else c for c in raw.columns]
if "Unnamed: 2" in raw.columns:
    raw = raw.rename(columns={"Unnamed: 2": "notes"})
elif len(raw.columns) == 2:
    raw["notes"] = ""

survey_time = raw.rename(
    columns={
        "Country": "country_wave",
        "Date": "field_period_raw",
    }
).copy()

print(survey_time.shape)
display(survey_time)

(21, 3)


,country_wave,field_period_raw,notes
0,Cambodia (2013),2/10/2013 - 3/22/2013,
1,Colombia (2018),08/01/2018 - 10/24/2018,
2,Cotedivoire (2019),6/8/2018 - 9/3/2018,
3,El Salvador (2018),11/1/2017-12/16/2017,
4,Eswatini (2007),5/15/2007 - 6/16/2007,
5,Eswatini (2022),4/1/2022 - 8/1/2022,
6,Haiti (2012),4/1/2012 - 6/1/2012,
7,Honduras (2017),June 2017,
8,Kenya (2010),11/25/2010 - 12/31/2010,
9,Kenya(2019),12/1/2018 - 1/31/2019,12/1/2018 - 1/31/2019 ( VIOLENCE AGAINST CHILD...


## §2 — Parse dates and flag non-machine-readable rows

**Strategy (v1):** Try a small set of separators (`–`, `-`, `to`) for ranges; flag **month-name-only** strings (pandas may otherwise parse bare month names to spurious ancient dates). Rows that need manual **start** / **end** for satellite windows stay flagged for PI/guide follow-up.

In [3]:
RANGE_SPLIT = re.compile(r"\s*(?:-|–|—|to)\s*", re.I)
# Two month names + a year (e.g. September–October 2013) — do not split on inner dash only
MONTH_NAME = (
    "january|february|march|april|may|june|july|august|september|october|november|december"
)
MONTH_PAIR_YEAR = re.compile(
    rf"(?i)^\s*({MONTH_NAME})\s*[-–—]\s*({MONTH_NAME})\s+(\d{{4}})\s*$"
)


def split_range(s: str) -> tuple[str | None, str | None]:
    s = (s or "").strip()
    if not s:
        return None, None
    parts = [p.strip() for p in RANGE_SPLIT.split(s, maxsplit=1)]
    if len(parts) == 2:
        return parts[0], parts[1]
    return s, None


def try_parse_date(txt: str | None) -> pd.Timestamp:
    if txt is None or not str(txt).strip():
        return pd.NaT
    t = pd.to_datetime(txt, errors="coerce", dayfirst=False)
    if pd.isna(t):
        t = pd.to_datetime(txt, errors="coerce", dayfirst=True)
    return t


starts: list[pd.Timestamp] = []
ends: list[pd.Timestamp] = []
parse_flag: list[str] = []

for _, row in survey_time.iterrows():
    raw_txt = str(row.get("field_period_raw", "")).strip()
    low = raw_txt.lower()
    if not raw_txt:
        starts.append(pd.NaT)
        ends.append(pd.NaT)
        parse_flag.append("empty")
    elif "june" in low and "september" in low:
        starts.append(pd.NaT)
        ends.append(pd.NaT)
        parse_flag.append("month_name_range_manual")
    elif re.match(r"^[a-z]+\s+\d{4}$", low):
        starts.append(pd.NaT)
        ends.append(pd.NaT)
        parse_flag.append("month_year_only_manual")
    elif "october" in low and "february" in low:
        starts.append(pd.NaT)
        ends.append(pd.NaT)
        parse_flag.append("cross_year_month_manual")
    elif MONTH_PAIR_YEAR.match(raw_txt):
        starts.append(pd.NaT)
        ends.append(pd.NaT)
        parse_flag.append("month_name_pair_manual")
    else:
        a, b = split_range(raw_txt)
        ts_a, ts_b = try_parse_date(a), try_parse_date(b)
        if b is None and pd.notna(ts_a):
            starts.append(ts_a)
            ends.append(ts_a)
            parse_flag.append("single_day")
        elif pd.notna(ts_a) and pd.notna(ts_b):
            starts.append(min(ts_a, ts_b))
            ends.append(max(ts_a, ts_b))
            parse_flag.append("range_ok")
        else:
            starts.append(ts_a)
            ends.append(ts_b)
            parse_flag.append("needs_review")

survey_time["field_start"] = starts
survey_time["field_end"] = ends
survey_time["date_parse_flag"] = parse_flag

# Guard: pandas can parse bare month names to year 0001 — invalidate those rows
BAD_YEAR = 1900


def _year(ts: pd.Timestamp) -> int | None:
    if pd.isna(ts):
        return None
    return int(ts.year)


for i in range(len(survey_time)):
    ys, ye = _year(survey_time.at[i, "field_start"]), _year(survey_time.at[i, "field_end"])
    if ys is not None and ys < BAD_YEAR:
        survey_time.at[i, "field_start"] = pd.NaT
        survey_time.at[i, "field_end"] = pd.NaT
        survey_time.at[i, "date_parse_flag"] = "month_name_pair_manual"
    elif ye is not None and ye < BAD_YEAR:
        survey_time.at[i, "field_start"] = pd.NaT
        survey_time.at[i, "field_end"] = pd.NaT
        survey_time.at[i, "date_parse_flag"] = "month_name_pair_manual"

display(survey_time[["country_wave", "field_period_raw", "field_start", "field_end", "date_parse_flag", "notes"]].sort_values("date_parse_flag"))

,country_wave,field_period_raw,field_start,field_end,date_parse_flag,notes
12,Moldova (2013),October 2018 – February 2019,NaT,NaT,cross_year_month_manual,DataUserGuide (may use online source)
11,Malawi (2013),September–October 2013,NaT,NaT,month_name_pair_manual,DataUserGuide (may use online source)
10,Letsotho (2018),June–September 2018,NaT,NaT,month_name_range_manual,DataUserGuide (may use online source)
7,Honduras (2017),June 2017,NaT,NaT,month_year_only_manual,
18,Zambia (2014),08/04/2014 - 10/05/2014,2014-08-04,2014-10-05,range_ok,dta
17,Tanzania (2024),03/01/2024 - 06/01/2024,2024-03-01,2024-06-01,range_ok,DataUserGuide (may use online source)
16,Tanzania (2009),11/06/2009 - 12/05/2009,2009-11-06,2009-12-05,range_ok,DataUserGuide
15,Nigeria (2014),5/01/2014 - 07/09/2014,2014-05-01,2014-07-09,range_ok,dta
14,Namibia (2019),3/22/2019 - 06/04/2019,2019-03-22,2019-06-04,range_ok,dta
13,Mozambique (2019),July 2019 - September 2019,2019-07-01,2019-09-01,range_ok,DataUserGuide (may use online source)


## §3 — Preliminary linkage keys

**Goal:** Stable keys for joining satellite extracts (admin polygons, FIRMS CSV pulls, raster zonal stats, etc.).

- **`iso3`:** ISO 3166-1 alpha-3 for API and boundary packages (manual map below — extend if CSV adds countries).
- **`country_base`:** Strip wave year in parentheses for grouping (heuristic).
- **Geo (later):** Link to harmonized **EA / cluster / admin** columns from Step-1 notebooks or external shapefiles — not loaded here in v1.

In [4]:
WAVE_YEAR = re.compile(r"\((\d{4})\)\s*$")


def country_base(name: str) -> str:
    s = str(name).strip()
    s = re.sub(r"\s*\(\d{4}\)\s*$", "", s).strip()
    return s


def wave_year(name: str) -> int | None:
    m = WAVE_YEAR.search(str(name).strip())
    return int(m.group(1)) if m else None


# Manual ISO3 for rows in VACS_survey_time.csv (correct typos in source labels here, not in CSV file)
NAME_TO_ISO3 = {
    "Cambodia": "KHM",
    "Colombia": "COL",
    "Cotedivoire": "CIV",
    "El Salvador": "SLV",
    "Eswatini": "SWZ",
    "Haiti": "HTI",
    "Honduras": "HND",
    "Kenya": "KEN",
    "Letsotho": "LSO",  # typo in CSV: Lesotho
    "Lesotho": "LSO",
    "Malawi": "MWI",
    "Moldova": "MDA",
    "Mozambique": "MOZ",
    "Namibia": "NAM",
    "Nigeria": "NGA",
    "Tanzania": "TZA",
    "Zambia": "ZMB",
    "Zimbabwe": "ZWE",
    "Rwanda": "RWA",
}


def lookup_iso3(country_wave: str) -> str | None:
    base = country_base(country_wave)
    # Handle "Kenya(2019)" style
    base = re.sub(r"([A-Za-z])\(", r"\1 (", base)
    base = country_base(base)
    return NAME_TO_ISO3.get(base)


survey_time["country_base"] = survey_time["country_wave"].map(country_base)
survey_time["wave_year"] = survey_time["country_wave"].map(wave_year)
survey_time["iso3"] = survey_time["country_wave"].map(lookup_iso3)

missing_iso = survey_time[survey_time["iso3"].isna()]
if len(missing_iso):
    print("Rows without ISO3 mapping — extend NAME_TO_ISO3:")
    display(missing_iso[["country_wave"]])
else:
    print("All rows mapped to iso3.")

display(survey_time[["country_wave", "iso3", "wave_year", "field_start", "field_end", "date_parse_flag"]])

All rows mapped to iso3.


,country_wave,iso3,wave_year,field_start,field_end,date_parse_flag
0,Cambodia (2013),KHM,2013.0,2013-02-10,2013-03-22,range_ok
1,Colombia (2018),COL,2018.0,2018-08-01,2018-10-24,range_ok
2,Cotedivoire (2019),CIV,2019.0,2018-06-08,2018-09-03,range_ok
3,El Salvador (2018),SLV,2018.0,2017-11-01,2017-12-16,range_ok
4,Eswatini (2007),SWZ,2007.0,2007-05-15,2007-06-16,range_ok
5,Eswatini (2022),SWZ,2022.0,2022-04-01,2022-08-01,range_ok
6,Haiti (2012),HTI,2012.0,2012-04-01,2012-06-01,range_ok
7,Honduras (2017),HND,2017.0,NaT,NaT,month_year_only_manual
8,Kenya (2010),KEN,2010.0,2010-11-25,2010-12-31,range_ok
9,Kenya(2019),KEN,2019.0,2018-12-01,2019-01-31,range_ok


## §4 — Derived fire products (scaffold)

As data research proceeds, append rows: **product id**, **sensor (MODIS/VIIRS/…)**, **spatial resolution**, **temporal extent vs survey**, **access URL or API**, **notes** (gaps, latency, known limitations).

Example columns (empty starter table):

In [ ]:
fire_products = pd.DataFrame(
    columns=[
        "product_name",
        "sensor",
        "provider",
        "spatial_res",
        "temporal_coverage_notes",
        "countries_verified",
        "link_or_collection_id",
        "notes",
    ]
)
fire_products